#### setup

In [ ]:
import sys, os
sys.path.insert(0, "../pointnet")

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

from pointnet_cls     import PointNetCls
from pointnet_dataset import ModelNet40

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = ModelNet40("../data", split="test", num_points=1024, augment=False)
print(f"classes: {dataset.classes}")
print(f"total samples: {len(dataset)}")

#### visualize raw point cloud

In [ ]:
def plot_cloud(points, title="point cloud", color_by="z"):
    """
    points: [N, 3] numpy
    color_by: 'z' colors by height
    """
    fig = plt.figure(figsize=(8,8))
    ax = fig.add_subplot(1,1,1, projection='3d')
    
    if color_by == "z":
        colors = points[:, 2]
        sc = ax.scatter(
            points[:, 0], points[:, 1], points[:, 2],
            c=colors, cmap='plasma', s=2, alpha=0.8
        )
        plt.colorbar(sc, ax=ax, shrink=0.5, label='z height')
    else:
        ax.scatter(
            points[:, 0], points[:, 1], points[:, 2],
            c='steelblue', s=2, alpha=0.8
        )
        
    ax.set_title(title, fontsize=14)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_zlabel('z')
    ax.grid(False)
    plt.tight_layout()
    plt.show()

In [ ]:
idx = 42
pts, label = dataset[idx]
class_name = dataset.classes[label.item()]

print(f"sample {idx}: class = {class_name} (label {label.item()})")
print(f"points shape: {pts.shape}")
print(f"xyz range — x: [{pts[:,0].min():.2f}, {pts[:,0].max():.2f}]  "
      f"y: [{pts[:,1].min():.2f}, {pts[:,1].max():.2f}]  "
      f"z: [{pts[:,2].min():.2f}, {pts[:,2].max():.2f}]")

plot_cloud(pts.numpy(), title=f"ground truth: {class_name}", color_by="z")

#### prediction using trained model

In [ ]:
ckpt_path = "../checkpoints/pointnet_cls_best.pth"

model = PointNetCls(num_classes=40, feature_transform=True).to(device)
ckpt = torch.load(ckpt_path, map_location=device)
model.load_state_dict(ckpt['state_dict'])
model.eval()